# SciPy 入门：统计、优化与金融计算

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Python-Scipy/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

## 一、安装与导入

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats, optimize, interpolate, integrate    #按需导入子模块

print(scipy.__version__)    #查看SciPy版本信息

### 概率分布

stats 模块里每一种分布都是一个对象，有一套统一的方法：

（示意代码，不在笔记本中运行）

```python
stats.norm.pdf(x, loc, scale)    #概率密度函数
stats.norm.cdf(x, loc, scale)    #累积分布函数：P(X <= x)
stats.norm.ppf(q, loc, scale)    #分位数函数：cdf的反函数
stats.norm.rvs(loc, scale, size)    #随机抽样
stats.norm.fit(data)    #用数据估计分布的参数
```

其中 `loc` 是位置参数（正态分布的均值），`scale` 是尺度参数（正态分布的标准差）。把 `norm` 换成 `t`、`chi2`、`lognorm` 等，就是其他分布，用法完全一样。NumPy 的随机数只能“抽”，而 SciPy 的分布对象还能算概率和分位数，这在金融中用得非常多：

In [ ]:
print("标准正态分布小于-1.96的概率：", round(stats.norm.cdf(-1.96), 4))
print("标准正态分布的1%分位数：", round(stats.norm.ppf(0.01), 4))
print("自由度为5的t分布的1%分位数：", round(stats.t.ppf(0.01, df=5), 4))

### 描述统计与正态性检验

和 [pandas 入门](https://wxrkz123.github.io/blog/Python-Pandas/)一样，先用 NumPy 模拟三只股票 2023 年的日收盘价：

In [ ]:
rng = np.random.default_rng(2119)    #固定随机种子，和pandas、matplotlib两篇文章的数据相同
dates = pd.bdate_range("2023-01-02", "2023-12-29")
daily_ret = rng.normal([0.0006, 0.0002, 0.0009], [0.018, 0.011, 0.026], size=(len(dates), 3))
prices = pd.DataFrame(100 * np.exp(daily_ret.cumsum(axis=0)),
                      index=dates, columns=["股票A", "股票B", "股票C"]).round(2)
ret = prices.pct_change().dropna()

d = stats.describe(ret["股票C"])
print(f"样本量 {d.nobs}，最小值 {d.minmax[0]:.4f}，最大值 {d.minmax[1]:.4f}")
print(f"均值 {d.mean:.5f}，方差 {d.variance:.6f}，偏度 {d.skewness:.3f}，峰度 {d.kurtosis:.3f}")

检验收益率是否服从正态分布，最常用的是 **Jarque-Bera 检验**，它同时检验偏度是否为 0、峰度是否为 3：

In [ ]:
jb = stats.jarque_bera(ret["股票C"])
print(f"模拟数据：JB统计量 = {jb.statistic:.2f}，p值 = {jb.pvalue:.3f}")

fat = stats.t.rvs(df=4, size=5000, random_state=3) * 0.015    #用t分布模拟“厚尾”的收益率
jb2 = stats.jarque_bera(fat)
print(f"厚尾数据：JB统计量 = {jb2.statistic:.2f}，p值 = {jb2.pvalue:.3g}")

### t 检验

股票 C 全年的平均日收益率是正的，但这能说明它的期望收益真的大于 0 吗？还是只是运气？用**单样本 t 检验**：

In [ ]:
for name in ret.columns:
    t = stats.ttest_1samp(ret[name], popmean=0)
    print(f"{name}：平均日收益率 {ret[name].mean():.4%}，t = {t.statistic:.2f}，p = {t.pvalue:.3f}")

### 求根：内部收益率与到期收益率

[NumPy 入门](https://wxrkz123.github.io/blog/Python-Numpy/)用 `npf.irr` 计算内部收益率，它的本质就是求方程 $NPV(r) = 0$ 的根。用 `optimize.brentq` 可以自己求解，它需要一个函数和一个包含根的区间：

In [ ]:
cash_flows = np.array([-1000, 300, 400, 400, 300])    #期初投资1000，之后四年的现金流

def npv(r):
    return np.sum(cash_flows / (1 + r) ** np.arange(len(cash_flows)))

irr = optimize.brentq(npv, -0.99, 1)    #在-99%到100%之间找使NPV=0的利率
print(f"内含报酬率：{irr:.4%}")
print(f"验证：NPV({irr:.4f}) = {npv(irr):.2e}")

同样的方法可以计算债券的**到期收益率**（YTM）：使债券未来现金流的现值等于当前价格的贴现率。一只面值 100 元、票面利率 3%、每年付息、还有 5 年到期的债券，现在的价格是 98 元：

In [ ]:
def bond_price(y, coupon=3, face=100, years=5):
    t = np.arange(1, years + 1)
    return np.sum(coupon / (1 + y) ** t) + face / (1 + y) ** years

ytm = optimize.brentq(lambda y: bond_price(y) - 98, 0, 1)
print(f"到期收益率：{ytm:.4%}")

### 最优化：最小方差组合

`optimize.minimize` 用来求一个函数的最小值，还可以加上约束条件：

（示意代码，不在笔记本中运行）

```python
optimize.minimize(fun, x0, method, bounds, constraints)
```

用它求三只股票的**最小方差组合**：权重加起来等于 1，不允许卖空（每个权重在 0 到 1 之间），让组合的方差最小：

In [ ]:
cov = ret.cov().values * 252    #年化协方差矩阵
mean = ret.mean().values * 252    #年化平均收益率

def port_vol(w):
    return np.sqrt(w @ cov @ w)

res = optimize.minimize(
    port_vol,
    x0=np.ones(3) / 3,    #从等权重开始
    method="SLSQP",
    bounds=[(0, 1)] * 3,    #不允许卖空
    constraints={"type": "eq", "fun": lambda w: w.sum() - 1},    #权重之和等于1
)
print("是否成功：", res.success)
print("最小方差组合的权重：", {name: round(float(x), 3) for name, x in zip(ret.columns, res.x)})
print(f"组合的年化波动率：{res.fun:.2%}（单只股票中最低的是 {np.sqrt(np.diag(cov)).min():.2%}）")

### 曲线拟合：收益率曲线

其中 $\tau$ 是期限，$\beta_0$ 代表长期利率水平，$\beta_1$ 代表短期与长期利率之差，$\beta_2$ 代表曲线中段的弯曲程度。

In [ ]:
maturity = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 30])    #期限（年）
yields = np.array([1.40, 1.48, 1.58, 1.80, 1.95, 2.12, 2.28, 2.35, 2.55])    #收益率（%），示意数据

def nelson_siegel(tau, b0, b1, b2, lam):
    x = tau / lam
    f1 = (1 - np.exp(-x)) / x
    return b0 + b1 * f1 + b2 * (f1 - np.exp(-x))

params, _ = optimize.curve_fit(nelson_siegel, maturity, yields, p0=[2.5, -1, 1, 2])
print("参数：", {name: round(float(x), 3) for name, x in zip(["b0", "b1", "b2", "lambda"], params)})
print("拟合误差（基点）：", ((nelson_siegel(maturity, *params) - yields) * 100).round(1))

## 四、interpolate：插值

市场上只有几个关键期限的收益率，如果需要 4 年期、15 年期的收益率怎么办？最简单的方法是**插值**：

In [ ]:
linear = interpolate.interp1d(maturity, yields)    #线性插值
spline = interpolate.CubicSpline(maturity, yields, bc_type="natural")    #三次样条插值（自然边界条件）

for tau in [4, 15, 20]:
    print(f"{tau}年期：线性插值 {float(linear(tau)):.3f}%，三次样条 {float(spline(tau)):.3f}%，"
          f"Nelson-Siegel {nelson_siegel(tau, *params):.3f}%")

把三种方法画在一起比较（画图的方法见 [Matplotlib 入门](https://wxrkz123.github.io/blog/Python-Matplotlib/)）：

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "WenQuanYi Zen Hei"]
plt.rcParams["axes.unicode_minus"] = False

grid = np.linspace(0.25, 30, 300)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(grid, linear(grid), color="grey", ls="--", label="线性插值")
ax.plot(grid, spline(grid), color="tab:blue", label="三次样条插值")
ax.plot(grid, nelson_siegel(grid, *params), color="tab:red", label="Nelson-Siegel拟合")
ax.scatter(maturity, yields, color="black", zorder=3, label="市场报价（示意）")
ax.set_xlabel("期限（年）")
ax.set_ylabel("收益率（%）")
ax.set_title("收益率曲线：插值与拟合")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 五、integrate：数值积分

`integrate.quad` 计算一元函数的定积分：

（示意代码，不在笔记本中运行）

```python
integrate.quad(func, a, b)    #计算func在[a, b]上的积分，返回(积分值, 误差估计)
```

金融中的一个例子：期权到期时的期望收益，就是收益函数乘以概率密度后的积分。比如，假设股票到期价格服从对数正态分布，看涨期权的期望收益 $E[\max(S_T - K, 0)]$：

In [ ]:
S0, K, r, sigma, T = 100, 105, 0.02, 0.25, 1    #现价、行权价、无风险利率、波动率、期限（年）
ST_dist = stats.lognorm(s=sigma * np.sqrt(T), scale=S0 * np.exp((r - sigma**2 / 2) * T))    #风险中性下的到期价格分布

payoff, err = integrate.quad(lambda s: (s - K) * ST_dist.pdf(s), K, np.inf)    #只有S>K时才有收益
print(f"到期期望收益：{payoff:.4f}（误差估计 {err:.1e}）")
print(f"贴现到今天的期权价格：{np.exp(-r * T) * payoff:.4f}")

## 六、Black-Scholes 期权定价与隐含波动率

其中 $N(\cdot)$ 是标准正态分布的累积分布函数，正好就是 `stats.norm.cdf`：

In [ ]:
def bs_call(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

print(f"Black-Scholes公式：{bs_call(S0, K, r, sigma, T):.4f}")

反过来，如果市场上这个期权的价格是 10 元，它隐含的波动率是多少？这就是**隐含波动率**：使 Black-Scholes 价格等于市场价格的波动率。公式没法直接反解，又是一个求根问题：

In [ ]:
market_price = 10
iv = optimize.brentq(lambda s: bs_call(S0, K, r, s, T) - market_price, 0.001, 3)
print(f"隐含波动率：{iv:.2%}")

## 七、最大夏普比率组合

第三节求了最小方差组合。更常用的是**夏普比率最大**的组合：每承担一单位风险，获得的超额收益最多。`minimize` 只能求最小值，所以最小化夏普比率的**相反数**：

In [ ]:
rf = 0.02    #无风险利率

def neg_sharpe(w):
    return -(w @ mean - rf) / port_vol(w)

res2 = optimize.minimize(neg_sharpe, x0=np.ones(3) / 3, method="SLSQP",
                         bounds=[(0, 1)] * 3, constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
w = res2.x
print("最大夏普比率组合的权重：", {name: round(float(x), 3) for name, x in zip(ret.columns, w)})
print(f"年化收益率 {w @ mean:.2%}，年化波动率 {port_vol(w):.2%}，夏普比率 {-res2.fun:.3f}")

## 八、三种方法计算 VaR

最后，用 stats 模块计算股票 C 的 99% 单日**风险价值**（VaR）：在 99% 的概率下，一天最多亏多少。

In [ ]:
r = ret["股票C"]
var_hist = -np.percentile(r, 1)    #历史模拟法：直接取历史收益率的1%分位数
var_norm = -stats.norm.ppf(0.01, loc=r.mean(), scale=r.std())    #参数法：假设正态分布
nu, loc, scale = stats.t.fit(r)    #用数据估计t分布的参数
var_t = -stats.t.ppf(0.01, nu, loc, scale)    #参数法：假设t分布

print(f"历史模拟法：{var_hist:.2%}")
print(f"正态分布法：{var_norm:.2%}")
print(f"t分布法：  {var_t:.2%}（估计的自由度 {nu:.1f}）")

对真实的股票收益率，估计出的自由度通常只有 3 到 6，这时正态分布法会明显低估 VaR。可以用第二节那组用 t 分布生成的厚尾数据试一试：

In [ ]:
nu2, loc2, scale2 = stats.t.fit(fat)
print(f"厚尾数据：历史模拟法 {-np.percentile(fat, 1):.2%}，"
      f"正态分布法 {-stats.norm.ppf(0.01, fat.mean(), fat.std()):.2%}，"
      f"t分布法 {-stats.t.ppf(0.01, nu2, loc2, scale2):.2%}（自由度 {nu2:.1f}）")